# Grid of Caplets in the 3-Factor Cheyette Model — PINN Benchmark

**One calibrated market, many derivatives.** A single Physics-Informed Neural
Network learns the pricing map
$$(t,\; x_1,x_2,x_3,x_4;\; T_C,\; K) \;\longrightarrow\; \text{caplet price}$$
across a grid of maturities and strikes, by solving the 4D+time valuation PDE
(Beyna, Chiarella & Kang 2012, Theorem 7.3). The curvature-aware optimizers
(**SSBroyden / SSBFGS**) from the CrunchOptimizer fork drive it to high accuracy.

**Accuracy metric.** Price error normalized by a **single fixed reference ATM
Normal (Bachelier) Vega** — the Vega of the $T_C{=}1, T_B{=}2$ ATM caplet
($\approx 0.36098$) — applied identically to every contract. This is *not* a
per-contract Vega; using one constant makes the **0.1 bps** target a single
uniform price tolerance across the whole grid.

> This notebook documents the setup and runs a short CPU demonstration. The full
> sub-0.1-bps results require a GPU (H200); run `cheyette_caplet_grid_h200.py`
> there with the large default settings.


## 1. The market and the contract grid

The model dynamics are fixed by the paper's Table 2 calibration. Only the terminal payoff varies per contract.

In [ ]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)

from cheyette_analytic import PARAMS, F0, caplet_price
from cheyette_grid import (MATURITIES, STRIKES, VEGA_REF, BUDGET_01BPS,
                           build_grid, grid_arrays)

print("Calibrated 3-factor params (Table 2):")
for k, v in PARAMS.items():
    print(f"  {k:>6} = {v}")
print(f"\nInitial forward f0 = {F0:.2%}")
print(f"\nGrid: {len(MATURITIES)} maturities x {len(STRIKES)} strikes "
      f"= {len(MATURITIES)*len(STRIKES)} contracts")
print(f"Single reference ATM Normal Vega = {VEGA_REF:.8f}")
print(f"0.1 bps  <=>  price error <= {BUDGET_01BPS:.3e}  (uniform, all contracts)")


### The analytical oracle

We price every contract in closed form. This pricer reproduces the paper's Table 5 benchmark prices to ~1e-7, so it is a trustworthy ground truth.

In [ ]:
rows = build_grid()
print(f"{'TC':>4}{'TB':>4}{'K':>6}{'moneyness':>10}{'analytic price':>16}")
for r in rows:
    money = ("ITM" if r["K"] < r["fwd"] else
             "ATM" if abs(r["K"]-r["fwd"]) < 0.005 else "OTM")
    print(f"{r['TC']:>4.1f}{r['TB']:>4.1f}{r['K']:>6.0%}{money:>10}{r['price']:>16.6f}")


## 2. The valuation PDE (Theorem 7.3)

The caplet price $g(t,x)$ solves, for **every** contract (the operator is
contract-independent; only the terminal condition differs):

$$
\frac{\partial g}{\partial t}
+ \sum_{i=1}^4 b_i(t,x)\,\frac{\partial g}{\partial x_i}
+ \tfrac12 \sum_{i,j=1}^4 [\sigma\sigma^\top]_{ij}(t)\,
   \frac{\partial^2 g}{\partial x_i \partial x_j}
- r(t,x)\, g = 0,
\qquad r(t,x) = f_0 + \sum_i x_i .
$$

The drift $b_i$ is built from the deterministic $V_{ij}^{(k)}(t)$ functions
(Appendix A.1); the diffusion $\sigma\sigma^\top$ is **state-independent** with a
dense $2\times2$ block coupling $(x_1,x_2)$ and independent diagonal entries for
$x_3, x_4$. Let's look at the diffusion structure.

In [ ]:
from cheyette_vij import sigma_sigmaT, drift_b
import jax.numpy as jnp

t_demo = jnp.array(1.0)
sst = np.array(sigma_sigmaT(t_demo, {k: float(v) for k, v in PARAMS.items()}))
print("sigma sigma^T at t=1.0:")
np.set_printoptions(precision=6, suppress=True)
print(sst)
print("\nNote: dense (x1,x2) block, decoupled x3 and x4 -- exactly the paper's structure.")


## 3. The terminal condition (per contract)

At each contract's fixing time $T_C$, the caplet value is the discounted payoff
(Section 7.3.2):
$$\Phi = B(T_C,T_B)\,\max\!\big(R(T_C,T_B) - K,\, 0\big),$$
with the LIBOR rate $R$ and discount bond $B(T_C,T_B)$ from the analytical bond
formula (Lemma 4.2). The $\max(\cdot,0)$ **kink** is the hard spot the paper
flags; we round it with a softplus whose width is chosen so the induced price
bias ($\sim 6\times10^{-7}$) stays well under the $3.6\times10^{-6}$ budget.

## 4. Why the single reference Vega is the right cross-contract metric

The same absolute price tolerance is much tighter in *relative* terms for cheap OTM contracts — but in Vega/vol units the 0.1 bps bar is uniform, which is the whole point.

In [ ]:
print(f"Fixed price budget (0.1 bps, ALL contracts): {BUDGET_01BPS:.3e}\n")
print(f"{'TC':>4}{'K':>6}{'price':>12}{'rel budget':>12}")
for r in rows:
    print(f"{r['TC']:>4.1f}{r['K']:>6.0%}{r['price']:>12.6f}"
          f"{r['budget_01bps']/r['price']:>12.2%}")


## 5. Short CPU demonstration

We run a **small** grid PINN here just to show the price converges toward the
analytic values across all contracts simultaneously. This uses tiny batches and
few steps — it will *not* reach 0.1 bps on CPU. For the real run, use the H200
defaults.

The cell below shells out to the benchmark script with reduced settings.

In [ ]:
import subprocess, os, sys
env = dict(os.environ,
           WIDTH="48", DEPTH="4",
           N_INT="3000", N_TERM="1500",
           ADAM_STEPS="1500", PRINT_EVERY="500", RESAMPLE_EVERY="1000",
           BENCH_OPT="none", MAX2="0")
# (On CPU each step is ~0.25s; 1500 steps ~ 6 min. Reduce ADAM_STEPS to go faster.)
proc = subprocess.run([sys.executable, "-u", "cheyette_caplet_grid_h200.py"],
                      env=env, capture_output=True, text=True, timeout=1800)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-1000:])


## 6. Running the full benchmark on an H200

Install (CUDA 12):

```bash
pip install -U "jax[cuda12]" equinox optax matplotlib scipy
pip install "git+https://github.com/raj-brown/optimistix.git@SSBFGS"
```

Run, benchmarking both curvature-aware optimizers:

```bash
BENCH_OPT=both XLA_PYTHON_CLIENT_PREALLOCATE=false python cheyette_caplet_grid_h200.py
```

The script reports, for each stage (Adam, SSBroyden, SSBFGS), the **max** and
**mean** vol error in bps across the grid and how many of the 15 contracts pass
the 0.1 bps bar, plus a per-contract table. Results are saved to
`cheyette_caplet_grid_results.npz`.

### What to watch on the first real run
- **Whether 0.1 bps is reachable across the whole grid is an open empirical
  question** — and harder than the single contract, since one network must fit
  all maturities and strikes at once. The OTM short-maturity contracts are the
  binding constraint.
- Confirm the kink-smoothing bias by halving `EPS_RATE`; if any converged price
  shifts by more than ~1e-6, tighten it.
- The reported bps are in **Normal (Bachelier)** vol, normalized by the single
  reference Vega — not the Black vol the paper's tables use.


## 7. Visualizing the result (after a run)

Once `cheyette_caplet_grid_results.npz` exists (from a real run), this cell draws
the vol-error surface across the maturity/strike grid.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

if os.path.exists("cheyette_caplet_grid_results.npz"):
    d = np.load("cheyette_caplet_grid_results.npz")
    contracts = d["contracts"]; vb = d["best_vol_bps"]
    TCs = sorted(set(contracts[:, 0])); Ks = sorted(set(contracts[:, 2]))
    grid = np.full((len(TCs), len(Ks)), np.nan)
    for (tc, tb, k), e in zip(contracts, vb):
        grid[TCs.index(tc), Ks.index(k)] = e
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.imshow(grid, aspect="auto", origin="lower", cmap="viridis",
                   extent=[min(Ks)*100, max(Ks)*100, min(TCs), max(TCs)])
    ax.set_xlabel("strike (%)"); ax.set_ylabel(r"$T_C$ (yrs)")
    ax.set_title(f"Vol error (bps) — best run: {str(d['best_name'])}")
    for i, tc in enumerate(TCs):
        for j, k in enumerate(Ks):
            ax.text(k*100, tc, f"{grid[i,j]:.2f}", ha="center", va="center",
                    color="w", fontsize=8)
    fig.colorbar(im, ax=ax, label="vol error (bps)")
    plt.tight_layout(); plt.savefig("grid_vol_error.png", dpi=140)
    plt.show()
    print(f"max {vb.max():.4f} bps | mean {vb.mean():.4f} bps | "
          f"pass {(vb<=0.1).sum()}/{len(vb)}")
else:
    print("No results file yet. Run the benchmark first (Section 6).")
